In [ ]:
# next part
distribution_metrics = pd.DataFrame({
    "Skewness":
        df[numeric_cols].apply(skew),

    "Kurtosis":
        df[numeric_cols].apply(kurtosis)
})

display(distribution_metrics)
normality_results = []

test_columns = numeric_cols[:3]

for col in test_columns:

    stat,p = normaltest(
        df[col].dropna()
    )

    interpretation = (
        "Normally Distributed"
        if p > 0.05
        else "Not Normally Distributed"
    )

    normality_results.append([
        col, stat, p, interpretation
    ])

normality_df = pd.DataFrame(
    normality_results,
    columns=[
        "Column",
        "Statistic",
        "P_Value",
        "Interpretation"
    ]
)

display(normality_df)
iqr_results = []

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    count = (
        (df[col] < lower) |
        (df[col] > upper)
    ).sum()

    iqr_results.append([
        col,
        count
    ])

iqr_df = pd.DataFrame(
    iqr_results,
    columns=[
        'Column',
        'IQR_Outliers'
    ]
)

display(iqr_df)
zscore_results = []

for col in numeric_cols:

    z = np.abs(
        stats.zscore(
            df[col].fillna(
                df[col].median()
            )
        )
    )

    count = (z > 3).sum()

    zscore_results.append([
        col,
        count
    ])

zscore_df = pd.DataFrame(
    zscore_results,
    columns=[
        "Column",
        "Zscore_Outliers"
    ]
)

display(zscore_df)
outlier_comparison = pd.merge(
    iqr_df,
    zscore_df,
    on='Column'
)

display(outlier_comparison)
if 'final_price' in df.columns:

    Q1 = df['final_price'].quantile(0.25)
    Q3 = df['final_price'].quantile(0.75)

    IQR = Q3-Q1

    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR

    final_price_outliers = df[
        (df['final_price']<lower) |
        (df['final_price']>upper)
    ]

    top10_outliers = (
        final_price_outliers
        .sort_values(
            by='final_price',
            ascending=False
        )
        .head(10)
    )

    display(top10_outliers)

    top10_outliers.to_csv(
        "outlier_detection.csv",
        index=False
    )
pearson_corr = (
    df[numeric_cols]
    .corr(method='pearson')
)

display(pearson_corr)

plt.figure(figsize=(12,8))
sns.heatmap(
    pearson_corr,
    annot=True,
    cmap='coolwarm'
)
plt.title("Pearson Correlation")
plt.show()
spearman_corr = (
    df[numeric_cols]
    .corr(method='spearman')
)

display(spearman_corr)
corr_matrix = (
    pearson_corr
    .abs()
    .unstack()
    .sort_values(
        ascending=False
    )
)

corr_matrix = corr_matrix[
    corr_matrix < 1
]

strong_corr = corr_matrix[
    corr_matrix > 0.3
]

strong_corr = strong_corr.reset_index()

strong_corr.columns = [
    'Variable1',
    'Variable2',
    'Correlation'
]

strong_corr = strong_corr.drop_duplicates()

display(strong_corr)

strong_corr.to_csv(
    "strong_correlations.csv",
    index=False
)
if len(numeric_cols) >= 2:

    plt.figure(figsize=(8,6))

    sns.scatterplot(
        data=df,
        x=numeric_cols[0],
        y=numeric_cols[1]
    )

    plt.title(
        f"{numeric_cols[0]} vs {numeric_cols[1]}"
    )

    plt.show()
if (
    'product_category' in df.columns and
    'final_price' in df.columns
):

    category_analysis = (
        df.groupby('product_category')
        .agg(
            Revenue=('final_price','sum'),
            Avg_Order_Value=('final_price','mean'),
            Transactions=('final_price','count')
        )
        .sort_values(
            by='Revenue',
            ascending=False
        )
    )

    display(category_analysis)
if (
    'region' in df.columns and
    'device_type' in df.columns and
    'final_price' in df.columns
):

    revenue_region_device = (
        df.groupby(
            ['region','device_type']
        )['final_price']
        .sum()
        .reset_index()
    )

    display(revenue_region_device)
customer_col = None

possible_cols = [
    'customer_type',
    'is_returning_customer',
    'returning_customer'
]

for col in possible_cols:
    if col in df.columns:
        customer_col = col
        break

if customer_col:

    customer_revenue = (
        df.groupby(customer_col)
        ['final_price']
        .sum()
    )

    customer_share = (
        customer_revenue /
        customer_revenue.sum()
    )*100

    display(customer_share)
if 'final_price' in df.columns:

    df['Customer_Segment'] = pd.qcut(
        df['final_price'],
        4,
        labels=[
            'Low Value',
            'Medium Value',
            'High Value',
            'Premium'
        ]
    )

    segment_table = (
        df.groupby(
            'Customer_Segment'
        )
        .agg(
            Customers=('final_price','count'),
            Revenue=('final_price','sum'),
            Avg_Order_Value=('final_price','mean')
        )
    )

    display(segment_table)

    segment_table.to_csv(
        "segment_analysis.csv"
    )
date_col = None

possible_date_cols = [
    'transaction_date',
    'date',
    'order_date',
    'purchase_date'
]

for col in possible_date_cols:
    if col in df.columns:
        date_col = col
        break

if date_col:

    df[date_col] = pd.to_datetime(
        df[date_col]
    )

    print("Date Column:", date_col)
if date_col:

    daily_revenue = (
        df.groupby(
            df[date_col].dt.date
        )['final_price']
        .sum()
        .reset_index()
    )

    daily_revenue.columns = [
        'Date',
        'Revenue'
    ]

    daily_revenue['MA7'] = (
        daily_revenue['Revenue']
        .rolling(7)
        .mean()
    )

    daily_revenue['MA14'] = (
        daily_revenue['Revenue']
        .rolling(14)
        .mean()
    )

    display(daily_revenue.head())
if date_col:

    df['Day_of_Week'] = (
        df[date_col]
        .dt.day_name()
    )

    day_analysis = (
        df.groupby(
            'Day_of_Week'
        )['final_price']
        .sum()
        .sort_values(
            ascending=False
        )
    )

    display(day_analysis)
if date_col:

    plt.figure(figsize=(14,6))

    plt.plot(
        daily_revenue['Date'],
        daily_revenue['Revenue'],
        label='Revenue'
    )

    plt.plot(
        daily_revenue['Date'],
        daily_revenue['MA7'],
        label='7 Day MA'
    )

    plt.plot(
        daily_revenue['Date'],
        daily_revenue['MA14'],
        label='14 Day MA'
    )

    plt.legend()

    plt.title(
        "Revenue Trend"
    )

    plt.xticks(rotation=45)

    plt.show()
if date_col:

    df['Transaction_Month'] = (
        df[date_col].dt.month
    )

    df['Transaction_Weekday'] = (
        df[date_col].dt.dayofweek
    )

if (
    'final_price' in df.columns and
    'quantity' in df.columns
):

    df['Revenue_per_Unit'] = (
        df['final_price'] /
        df['quantity']
    )

display(df.head())
#MULTIINDEX WEEKLY SALES

if (
    date_col and
    'product_category' in df.columns
):

    weekly_sales = (
        df.groupby([
            pd.Grouper(
                key=date_col,
                freq='W'
            ),
            'product_category'
        ])
        ['final_price']
        .sum()
    )

    display(weekly_sales)
#HIGH VALUE CUSTOMER SCORE


score_columns = []

for col in numeric_cols:
    if col != 'final_price':
        score_columns.append(col)

if len(score_columns) > 0:

    scaler = StandardScaler()

    scaled = scaler.fit_transform(
        df[score_columns]
        .fillna(0)
    )

    df['High_Value_Score'] = (
        scaled.mean(axis=1)
    )

    display(
        df[
            ['High_Value_Score']
        ].head()
    )
#LOGISTIC REGRESSION

target_col = None

for col in [
    'is_returning_customer',
    'returning_customer'
]:
    if col in df.columns:
        target_col = col
        break

if target_col:

    model_data = df.copy()

    model_data = pd.get_dummies(
        model_data,
        drop_first=True
    )

    X = model_data.drop(
        columns=[target_col],
        errors='ignore'
    )

    y = model_data[target_col]

    X = X.fillna(0)

    X_train,X_test,y_train,y_test = (
        train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )
    )

    model = LogisticRegression(
        max_iter=5000
    )

    model.fit(
        X_train,
        y_train
    )

    preds = model.predict(
        X_test
    )

    print(
        "Accuracy:",
        accuracy_score(
            y_test,
            preds
        )
    )

    print(
        classification_report(
            y_test,
            preds
        )
    )


In [ ]:
SELECT userid, COUNT(*) AS login_count
FROM attendance
GROUP BY userid
ORDER BY login_count DESC;

In [1]:
data = [
    ("aman", "java,python"),
    ("aman", "sql,airflow"),
    ("nikita", "it,finance"),
    ("nikita", "marketing,media"),
    ("sonu", "java,C++"),
]

In [2]:
for name, skills in data:
    skill_list = skills.split(",")

    for skill in skill_list:
        print(name, skill)

aman java
aman python
aman sql
aman airflow
nikita it
nikita finance
nikita marketing
nikita media
sonu java
sonu C++


In [6]:
import pandas as pd

In [7]:
data = {
    "userid": [101, 101, 102, 102, 102, 103, 103, 104],
    "login": [
        "2025-06-01 09:00:00",
        "2025-06-01 14:00:00",
        "2025-06-01 08:30:00",
        "2025-06-01 13:00:00",
        "2025-06-02 09:00:00",
        "2025-06-01 10:00:00",
        "2025-06-02 10:00:00",
        "2025-06-01 09:15:00",
    ],
    "logout": [
        "2025-06-01 12:00:00",
        "2025-06-01 18:00:00",
        "2025-06-01 12:00:00",
        "2025-06-01 17:00:00",
        "2025-06-02 18:00:00",
        "2025-06-01 18:00:00",
        "2025-06-02 18:00:00",
        "2025-06-01 17:30:00",
    ],
}

In [8]:
df = pd.DataFrame(data)
print(df)

   userid                login               logout
0     101  2025-06-01 09:00:00  2025-06-01 12:00:00
1     101  2025-06-01 14:00:00  2025-06-01 18:00:00
2     102  2025-06-01 08:30:00  2025-06-01 12:00:00
3     102  2025-06-01 13:00:00  2025-06-01 17:00:00
4     102  2025-06-02 09:00:00  2025-06-02 18:00:00
5     103  2025-06-01 10:00:00  2025-06-01 18:00:00
6     103  2025-06-02 10:00:00  2025-06-02 18:00:00
7     104  2025-06-01 09:15:00  2025-06-01 17:30:00


In [17]:
count = df["userid"].value_counts()

In [18]:
count

userid
102    3
101    2
103    2
104    1
Name: count, dtype: int64

In [19]:
second_highest = count.unique()[1]

In [20]:
second_highest

np.int64(2)

In [21]:
result = count[count == second_highest]

In [22]:
result

userid
101    2
103    2
Name: count, dtype: int64

In [ ]:
looks,dancnic, humar, taketative, confidence
9,8,7,6,8
5

